# Modul 5: Convolutional Neural Network Dasar

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  

Simpan berkas ini sebagai `M05_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. **Bagian A dikerjakan sebelum menulis kode.** Isi tabel shape dengan tangan.
3. Protokol tetap: subset $10\,000$/$2\,000$, Adam $10^{-3}$, batch $128$, $10$ epoch ($790$ update).
4. Bila memakai Fashion-MNIST karena keterbatasan CPU, sebutkan pada laporan — angka pembanding ikut berubah.
5. Catat seluruh run ke `metrics.csv`.
6. Luaran: `M05_NIM.ipynb`, `M05_NIM.pdf`, `M05_NIM_metrics.csv`, dan tabel arsitektur.

In [ ]:
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, FashionMNIST

DATASET = 'cifar10'          # ganti ke 'fashion' bila hanya tersedia CPU
NIM = 'TODO'                 # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE),
       'dataset': DATASET, 'seed': SEED})

## A. Pre-lab dan tabel shape - 15 poin

Kerjakan **sebelum** menulis kode apa pun.

1. **Arti keempat sumbu $N\times C\times H\times W$:** TODO
2. **Hasil $\lfloor(32+2\cdot1-3)/1\rfloor+1$ dan mengapa sama dengan masukannya:** TODO
3. **Jumlah parameter `Conv2d(3, 32, kernel_size=3)` termasuk bias:** TODO
4. **Mengapa parameter convolution tidak bergantung ukuran citra, sedangkan `Linear` bergantung:** TODO

**Tabel shape CNN baseline** untuk masukan $3\times32\times32$. Isi dengan tangan:

| Layer | Keluaran ($C\times H\times W$) | Parameter |
|---|---|---|
| `Conv2d(3,32,k=3,p=1)` | TODO | TODO |
| `MaxPool2d(2)` | TODO | 0 |
| `Conv2d(32,64,k=3,p=1)` | TODO | TODO |
| `MaxPool2d(2)` | TODO | 0 |
| `Linear(?, 128)` | TODO | TODO |
| `Linear(128, 10)` | TODO | TODO |
| **Total** | | **TODO** |

## B. Data - bagian dari 20 poin

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

if DATASET == 'cifar10':
    tr = CIFAR10(root=DATA_ROOT, train=True, download=False)
    te = CIFAR10(root=DATA_ROOT, train=False, download=False)
    X_penuh = torch.tensor(tr.data).permute(0, 3, 1, 2).float() / 255.0
    y_penuh = torch.tensor(tr.targets)
    X_uji_mentah = torch.tensor(te.data).permute(0, 3, 1, 2).float() / 255.0
    y_uji = torch.tensor(te.targets)
    KELAS = tr.classes
else:
    tr = FashionMNIST(root=DATA_ROOT, train=True, download=False)
    te = FashionMNIST(root=DATA_ROOT, train=False, download=False)
    X_penuh = tr.data.unsqueeze(1).float() / 255.0
    y_penuh = tr.targets
    X_uji_mentah = te.data.unsqueeze(1).float() / 255.0
    y_uji = te.targets
    KELAS = tr.classes

C_IN, H, W = X_penuh.shape[1:]

# TODO 1: ambil 10.000 latih dan 2.000 validasi terstratifikasi (random_state=SEED).
idx_latih, idx_val = ...

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]

# TODO 2: hitung MEAN dan STD PER KANAL dari subset latih saja.
#         Petunjuk: gunakan dim=(0, 2, 3) dan keepdim=True.
MEAN, STD = ..., ...
normalkan = lambda t: (t - MEAN) / STD

ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)
ds_uji = TensorDataset(normalkan(X_uji_mentah), y_uji)

print(f'citra {C_IN}x{H}x{W} | latih {len(ds_latih)} validasi {len(ds_val)}')
assert (len(ds_latih), len(ds_val)) == (10_000, 2_000)
assert torch.bincount(y_latih).min().item() == 1_000, 'subset harus terstratifikasi'
print('split sesuai protokol')

## C. CNN dua blok dan hitungan parameter - 20 poin

In [ ]:
def buat_cnn(c1=32, c2=64, k=3, c_in=C_IN, hw=H):
    """TODO 3: Conv-ReLU-Pool dua kali, lalu Flatten dan dua Linear.

    - padding dipilih agar ukuran bertahan (p = k // 2);
    - masukan Linear pertama = c2 * (hw // 4) ** 2;
    - selalu panggil seed_everything(SEED) lebih dahulu.
    """
    raise NotImplementedError

cnn = buat_cnn()
rincian = [(nama, sum(q.numel() for q in m.parameters()))
           for nama, m in cnn.named_children()
           if sum(q.numel() for q in m.parameters()) > 0]
total_cnn = sum(n for _, n in rincian)
for nama, n in rincian:
    print(f'  layer {nama:>2}: {n:>9,} ({100*n/total_cnn:5.1f}%)')
print('TOTAL CNN:', f'{total_cnn:,}')

if DATASET == 'cifar10':
    assert total_cnn == 545_098, 'arsitektur belum sesuai protokol CIFAR-10'
print('jumlah parameter terverifikasi')

In [ ]:
# TODO 4: verifikasi tabel Bagian A dengan forward hook.
#         Pasang hook pada setiap child module, jalankan satu batch (4, C_IN, H, W),
#         lalu tampilkan DataFrame berisi nama layer, jenis modul, dan shape keluaran.
raise NotImplementedError

**Kecocokan tabel.** Apakah seluruh shape hasil hitungan tangan cocok dengan hook? Bila ada selisih, jelaskan penyebabnya di sini:

TODO

**Letak parameter.** Berapa persen parameter berada pada `Linear` pertama, dan apa artinya? TODO

## D. FNN pembanding pada anggaran setara - 25 poin

Setel `hidden` FNN agar jumlah parameternya sedekat mungkin dengan CNN, sehingga selisih kinerja tidak dapat dijelaskan oleh ukuran model.

In [ ]:
def buat_fnn(hidden, c_in=C_IN, hw=H):
    """TODO 5: Flatten -> Linear(c_in*hw*hw, hidden) -> ReLU -> Linear(hidden, 10)."""
    raise NotImplementedError

# TODO 6: hitung `hidden` yang menyamakan anggaran parameter dengan CNN,
#         lalu cetak kedua jumlah parameter berdampingan.
hidden = ...
fnn = buat_fnn(hidden)
p_fnn = sum(p.numel() for p in fnn.parameters())
print(f'hidden={hidden}  parameter FNN={p_fnn:,}  CNN={total_cnn:,}')
assert abs(p_fnn - total_cnn) / total_cnn < 0.01, 'anggaran belum setara (selisih > 1%)'

In [ ]:
BATCH, EPOCH = 128, 10

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    """TODO 7: kembalikan (loss rata-rata, akurasi). Jangan lupa model.eval()."""
    raise NotImplementedError

def jalankan(model, label, epoch=EPOCH):
    """TODO 8: satu fungsi pelatihan untuk SELURUH run.

    Adam lr=1e-3, CrossEntropyLoss, catat val_loss dan val_acc tiap epoch.
    Kembalikan (riwayat, catatan) dengan catatan memuat run_id, seed,
    arsitektur, parameter, n_update, train_loss, val_loss, val_acc, gap,
    detik_per_epoch, runtime_s.
    """
    raise NotImplementedError

hasil, kurva = [], {}
for model, label in [(fnn, 'fnn'), (buat_cnn(), 'cnn-baseline')]:
    r, catatan = jalankan(model, label)
    hasil.append(catatan); kurva[label] = r

print(pd.DataFrame(hasil)[['run_id', 'parameter', 'n_update', 'val_loss',
                           'val_acc', 'gap', 'detik_per_epoch']].to_string(index=False))
assert {c['n_update'] for c in hasil} == {790} or DATASET != 'cifar10', \
    'kedua model harus memakai anggaran update yang sama'

In [ ]:
# TODO 9: tampilkan kurva validation accuracy kedua model dalam satu grafik berlabel.
raise NotImplementedError

**Penjelasan selisih.** Kedua model memakai parameter yang praktis sama. Apa yang menjelaskan selisih akurasinya? TODO

**Waktu per epoch.** Mana yang lebih lambat, dan mengapa hal itu tidak bertentangan dengan jumlah parameter yang sama? TODO

**Checkpoint menit ke-90.** Tunjukkan kepada asisten: tabel shape yang cocok dengan hook, kedua jumlah parameter, dan grafik perbandingan FNN--CNN.

## E. Dua varian terkendali - 15 poin

Satu varian, satu perubahan dari baseline.

In [ ]:
# TODO 10: jalankan varian A (k=5) dan varian B (c1=64, c2=128).
#          Cetak jumlah parameter tiap varian sebelum melatih.
raise NotImplementedError

tabel = pd.DataFrame(hasil)
print(tabel[['run_id', 'parameter', 'val_loss', 'val_acc', 'gap',
             'detik_per_epoch']].to_string(index=False))
assert len(tabel) == 4, 'harus ada empat run: fnn, baseline, varian A, varian B'

**Lonjakan parameter varian B.** Berapa parameternya, layer mana yang menyumbang paling banyak, dan mengapa? TODO

## F. Feature map dan error analysis - 15 poin

In [ ]:
# TODO 11: tampilkan delapan feature map pertama dari kedua blok convolution
#          untuk satu citra validasi. Beri keterangan blok dan indeks kanal.
raise NotImplementedError

In [ ]:
# TODO 12: tampilkan confusion matrix model terbaik pada validation set,
#          lalu pilih LIMA prediksi salah, tampilkan citranya, dan catat
#          label benar serta label prediksinya.
raise NotImplementedError

**Analisis lima kesalahan.** Untuk setiap contoh, tuliskan dugaan penyebabnya:

| No | Label benar | Prediksi | Dugaan penyebab |
|----|-------------|----------|-----------------|
| 1 | TODO | TODO | TODO |
| 2 | TODO | TODO | TODO |
| 3 | TODO | TODO | TODO |
| 4 | TODO | TODO | TODO |
| 5 | TODO | TODO | TODO |

In [ ]:
# TODO 13: simpan seluruh run ke metrics.csv.
tabel.insert(0, 'module', 'M05')
tabel.insert(1, 'student_id', NIM)
tabel.insert(2, 'dataset', DATASET)
tabel.to_csv(f'M05_{NIM}_metrics.csv', index=False)
print(f'{len(tabel)} baris tersimpan')

## G. Pertanyaan analisis - bagian dari 10 poin

1. Pada anggaran parameter yang praktis sama, model mana yang menang dan berapa selisihnya? TODO
2. Berapa persen parameter CNN ada di `Linear` pertama, dan apa akibatnya bila kanal blok kedua digandakan? TODO
3. Bandingkan waktu per epoch FNN dan CNN — mengapa CNN dapat lebih lambat meski parameternya tidak lebih banyak? TODO
4. Apa beda watak feature map blok pertama dan blok kedua? TODO
5. Dua kelas mana yang paling sering tertukar, dan apakah masuk akal secara visual? TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, device, dan dataset yang dipakai tercantum.
- [ ] Tabel shape Bagian A diisi **sebelum** kode dan selisihnya dijelaskan.
- [ ] Jumlah parameter CNN dan FNN tercetak dan lolos sel pemeriksaan.
- [ ] Keempat run memakai split, seed, dan anggaran update yang sama.
- [ ] Feature map diberi keterangan blok dan indeks kanal.
- [ ] Lima prediksi salah ditampilkan beserta dugaan penyebabnya.
- [ ] `metrics.csv` memuat keempat run.
- [ ] Notebook lolos *Restart Kernel and Run All*.